# Notebook 02.6: Stage 2 Head Hyperparameter Search (per backbone)
Searches attn_dim, dropout, lr, weight_decay for the Gated ABMIL head, once per architecture. 

Uses features from ONE fold-holdout Stage 1 backbone per architecture (fold 1's backbone), with an inner validation split that keeps fold 0 completely untouched 


In [ ]:
import os, json, time, gc
import numpy as np
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
print(f"optuna  : {optuna.__version__}")

In [ ]:
import sys
sys.path.append('/kaggle/input/datasets/mfjmrizvi/cbis-ddsm-project-config')
from abmil_common import (
    build_backbone, PatchClassifier, BagClassifier, CachedBagDataset,
    collate_cached, extract_features, get_normalisation_tensors,
)

In [ ]:
# Paths
NB02 = Path("/kaggle/input/notebooks/mfjmrizvi/02-mil-patch-extraction")
NB03 = Path("/kaggle/input/notebooks/mfjmrizvi/03-efficientnet-b0")  
NB04 = Path("/kaggle/input/notebooks/mfjmrizvi/04-convnext-nano")     
NB05 = Path("/kaggle/input/notebooks/mfjmrizvi/05-swint")            
OUT  = Path("/kaggle/working")

X_TRAIN_PATH       = NB02 / "X_train_patches.npy"
Y_TRAIN_PATH       = NB02 / "y_train_labels.npy"
BAG_IDS_TRAIN_PATH = NB02 / "bag_ids_train.npy"
FOLD_IDS_PATH      = NB02 / "fold_ids.npy"
CLASS_WEIGHTS_PATH = NB02 / "class_weights.json"

SEED           = 42
STAGE1_FOLD    = 0
INNER_VAL_FOLD = 1     # same rule as Step 1 — fold 0 never touches any search
PATCH_SIZE     = 224
N_TRIALS_STAGE2 = 20

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

_mean_gpu, _std_gpu = get_normalisation_tensors(DEVICE)

In [ ]:
# Load NB02 outputs
X_train_all  = np.load(X_TRAIN_PATH)
y_train_all  = np.load(Y_TRAIN_PATH)
bag_ids_all  = np.load(BAG_IDS_TRAIN_PATH)
fold_ids     = np.load(FOLD_IDS_PATH)

with open(CLASS_WEIGHTS_PATH) as f:
    raw_cw = json.load(f)
class_weight_dict = {int(k): float(v) for k, v in raw_cw.items()}

all_bags = np.unique(bag_ids_all)
print(f"Loaded: {X_train_all.shape} patches, {len(all_bags)} bags")

In [ ]:
def extract_search_features(model_name, stage1_ckpt_path, X_all, device, patch_size=224):
    backbone = build_backbone(model_name, pretrained=False).to(device)
    with torch.no_grad():
        _dummy = torch.zeros(2, 3, patch_size, patch_size, device=device)
        feat_dim = backbone(_dummy).shape[1]
    del _dummy

    patch_model = PatchClassifier(backbone, feat_dim).to(device)
    patch_model.load_state_dict(torch.load(stage1_ckpt_path, map_location=device))
    feature_extractor = patch_model.backbone
    feature_extractor.eval()
    for p in feature_extractor.parameters():
        p.requires_grad_(False)

    feats_all = extract_features(X_all, feature_extractor, _mean_gpu, _std_gpu, device)

    del backbone, patch_model, feature_extractor
    gc.collect(); torch.cuda.empty_cache()
    return feats_all, feat_dim

In [ ]:
# Stage 2 Optuna objective
#searches the attention head only, on cached features from the fold-1-holdout backbone.

def make_stage2_objective(feats_all, y_all, bag_ids_all_, search_train_bags, search_val_bags,
                           seed, max_epochs=15, patience=5):

    def objective(trial):
        torch.manual_seed(seed + trial.number)
        np.random.seed(seed + trial.number)

        attn_dim     = trial.suggest_categorical("attn_dim", [64, 128, 256])
        dropout      = trial.suggest_float("dropout", 0.0, 0.5)
        lr           = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
        weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)

        train_mask = np.isin(bag_ids_all_, search_train_bags)
        val_mask   = np.isin(bag_ids_all_, search_val_bags)

        train_ds = CachedBagDataset(feats_all[train_mask], y_all[train_mask],
                                     bag_ids_all_[train_mask], search_train_bags)
        val_ds   = CachedBagDataset(feats_all[val_mask], y_all[val_mask],
                                     bag_ids_all_[val_mask], search_val_bags)
        train_dl = DataLoader(train_ds, batch_size=1, shuffle=True, collate_fn=collate_cached)
        val_dl   = DataLoader(val_ds, batch_size=1, shuffle=False, collate_fn=collate_cached)

        feat_dim = feats_all.shape[1]
        model = BagClassifier(feat_dim, attn_dim, dropout=dropout, gated=True).to(DEVICE)
        criterion = nn.BCEWithLogitsLoss()
        optimiser = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

        best_val_loss, patience_ctr = float("inf"), 0

        for epoch in range(1, max_epochs + 1):
            model.train()
            for h_list, labels in train_dl:
                h, label = h_list[0].to(DEVICE), labels[0].to(DEVICE)
                optimiser.zero_grad()
                logit, _ = model(h)
                loss = criterion(logit, label)
                loss.backward()
                optimiser.step()

            model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for h_list, labels in val_dl:
                    h, label = h_list[0].to(DEVICE), labels[0].to(DEVICE)
                    logit, _ = model(h)
                    val_loss += criterion(logit, label).item()
            val_loss /= len(val_ds)

            trial.report(val_loss, epoch)
            if trial.should_prune():
                del model, optimiser
                gc.collect(); torch.cuda.empty_cache()
                raise optuna.TrialPruned()

            if val_loss < best_val_loss:
                best_val_loss, patience_ctr = val_loss, 0
            else:
                patience_ctr += 1
                if patience_ctr >= patience:
                    break

        del model, optimiser
        gc.collect(); torch.cuda.empty_cache()
        return best_val_loss

    return objective

In [ ]:
# Run the search: one per architecture
stage2_search_configs = [
    ("efficientnet_b0","effnet_b0", NB03 / "efficientnet_b0_stage1_fold1.pth", SEED + 0),
    ("convnext_nano", "convnext_nano", NB04 / "convnext_nano_stage1_fold1.pth", SEED + 1),
    ("swin_tiny_patch4_window7_224", "swin_t", NB05 / "swin_tiny_patch4_window7_224_stage1_fold1.pth", SEED + 2),
]

search_bags = all_bags[fold_ids[all_bags] != STAGE1_FOLD]           
search_train_bags = search_bags[fold_ids[search_bags] != INNER_VAL_FOLD]
search_val_bags   = search_bags[fold_ids[search_bags] == INNER_VAL_FOLD]

stage2_best_params = {}

for model_name, tag, ckpt_path, seed in stage2_search_configs:
    print(f"\n{'='*60}\nStage 2 Optuna search: {tag} (seed={seed}, {N_TRIALS_STAGE2} trials)\n{'='*60}")

    print(f"Extracting features from {ckpt_path.name}...")
    feats_all, feat_dim = extract_search_features(model_name, ckpt_path, X_train_all, DEVICE)
    print(f"Features shape: {feats_all.shape}")

    sampler = TPESampler(seed=seed)
    pruner  = MedianPruner(n_startup_trials=5, n_warmup_steps=3)
    study = optuna.create_study(direction="minimize", sampler=sampler, pruner=pruner)

    objective = make_stage2_objective(
        feats_all, y_train_all, bag_ids_all, search_train_bags, search_val_bags,
        seed=seed, max_epochs=15, patience=5
    )

    def save_partial(study, trial, tag=tag, model_name=model_name):
        partial = {
            "model_name": model_name,
            "best_params": study.best_params,
            "best_value": study.best_value,
            "n_trials_completed": len(study.trials),
            "n_trials_target": N_TRIALS_STAGE2,
        }
        with open(OUT / f"{tag}_stage2_partial.json", "w") as f:
            json.dump(partial, f, indent=2)

    t0 = time.time()
    study.optimize(objective, n_trials=N_TRIALS_STAGE2, callbacks=[save_partial])
    elapsed = (time.time() - t0) / 60

    print(f"{tag} Stage 2 search complete — {elapsed:.1f} min")
    print("Best params:", study.best_params)
    print("Best val_loss:", study.best_value)

    stage2_best_params[tag] = {
        "model_name": model_name,
        "seed": seed,
        "stage1_checkpoint_used": str(ckpt_path),
        "best_params": study.best_params,
        "best_value": study.best_value,
        "n_trials": N_TRIALS_STAGE2,
        "search_time_min": elapsed,
    }

    with open(OUT / f"{tag}_stage2_optuna_study.json", "w") as f:
        json.dump(stage2_best_params[tag], f, indent=2)

    del feats_all
    gc.collect(); torch.cuda.empty_cache()

print("\nAll Stage 2 searches complete:")
print(json.dumps(stage2_best_params, indent=2))